# 08 — Train single-channel u or v (L1 baseline or adv-NO)

Train **one field at a time** (`COMPONENT = "u"` or `"v"`) on exported single-channel data (`*_u_dt96_*`, `*_v_dt96_*`). Same training stack as [04_train_adv_no_fix_window.ipynb](04_train_adv_no_fix_window.ipynb).

**Switch in config cell:**
- `COMPONENT` — `"u"` | `"v"`
- `LOSS_MODE` — `"l1"` (masked L1 only) | `"adv_no"` (L1 + perceptual + adversarial)
- `LOSS_SPATIAL_MODE` — `"full"` | `"gaussian_center"` (set `LOSS_GAUSSIAN_SIGMA`)
- `LOSS_METRIC` — `"components"` (L1 on the exported channel; use this for single-channel)
- Perceptual: `PERCEPTUAL_BACKEND` = `"lpips"` | `"vgg"`; staging via `PERCEPTUAL_WARMUP_EPOCHS`, `WARMUP_EPOCHS`

**Outputs** under `Models/adv_no/{RUN_NAME}/`:
- `best_model.pth`, `history/`, `viz/curves/loss_overview.png`
- Sample PNGs: `viz/samples/` (04 style) or `viz/samples_gauss_circles/` (circle layout)
- Optional `viz/curves/circle_l1_overview.png` when gaussian-circle viz is enabled

Change `RUN_TAG` for each experiment. Set `PROTECT_EXISTING_RUN = False` to overwrite the same tag.

In [1]:
import importlib
import json
import sys
from dataclasses import asdict
from pathlib import Path

import src.train.adv_no_train as adv_no_train_module
importlib.reload(adv_no_train_module)
from src.train.adv_no_train import AdvNOTrainConfig, run_adv_no_training


In [4]:
DT = 96

# --- which field to train (u and v use separate RUN_NAME folders — never overwrite) ---
# --- u (completed) ---
# COMPONENT = "u"
# DATA_PREFIX = "data/sample_16_19_smallfixwin_u_dt96"
# CHANNEL_NAMES = ("u",)
# iD = 1
# RUN_BASE = "adv_no_u_16_19_dt96"
# RUN_TAG = "lpips_full_l1"
# RUN_NAME = f"{RUN_BASE}_{RUN_TAG}"  # -> Models/adv_no/adv_no_u_16_19_dt96_lpips_full_l1

# --- v (active) ---
COMPONENT = "v"
DATA_PREFIX = "data/sample_16_19_smallfixwin_v_dt96"
CHANNEL_NAMES = ("v",)
iD = 1
RUN_BASE = "adv_no_v_16_19_dt96"
RUN_TAG = "lpips_full_l1_v"  # same tag as u run; folder differs via RUN_BASE
RUN_NAME = f"{RUN_BASE}_{RUN_TAG}"  # -> Models/adv_no/adv_no_v_16_19_dt96_lpips_full_l1

RUN_DIR = Path("Models/adv_no") / RUN_NAME

# --- loss ---
LOSS_MODE = "l1"  # "l1" | "adv_no"
LOSS_METRIC = "components"  # single-channel: always components (not speed)
LOSS_SPATIAL_MODE = "full"  # "full" | "gaussian_center"
LOSS_GAUSSIAN_SIGMA = 32.0

# --- adv-NO (when LOSS_MODE == "adv_no") ---
BETA = 0.1
LAMBDA_PERC = 0.1
PERCEPTUAL_BACKEND = "lpips"  # "lpips" | "vgg"
LPIPS_NET = "alex"  # alex | vgg | squeeze (lpips only)
PERCEPTUAL_WEIGHTS = (1.0, 1.0, 1.0, 1.0, 1.0)  # vgg only
PERCEPTUAL_WARMUP_EPOCHS = 10
WARMUP_EPOCHS = 59

D_LR_RATIO = 0.5
D_SKIP_THRESHOLD = 0.2

# --- viz ---
VIZ_GAUSSIAN_CIRCLES = False  # True -> samples_gauss_circles/ + circle_l1 curves
VIZ_GAUSSIAN_CIRCLE_RADII = (16.0, 32.0, 64.0)
VIZ_GAUSSIAN_INNER_CIRCLE_RADII = (32.0, 16.0)
VIZ_GAUSSIAN_FIG_SCALE = 2.6
VIZ_RESIDUAL_METRIC = "l1"  # error panels: "l1" | "signed" | "l1_smooth"
VIZ_EVERY = 5
VIZ_SAMPLES = 1
VIZ_SEED = 42

USE_COSINE_LR = True
DETACHED = False
SHOW_INLINE_PLOTS = False
PREVIEW_LOSS_MASK = True
PROTECT_EXISTING_RUN = False
DEVICE = "cuda:7"

# Example presets (copy into RUN_TAG + settings above):
# gauss L1 only: LOSS_MODE=l1, LOSS_SPATIAL_MODE=gaussian_center, RUN_TAG=gauss_l1_circles
# adv-NO VGG: LOSS_MODE=adv_no, PERCEPTUAL_BACKEND=vgg, PERCEPTUAL_WARMUP_EPOCHS=21, WARMUP_EPOCHS=40


In [5]:
train_config = AdvNOTrainConfig(
    data_prefix=DATA_PREFIX,
    run_name=RUN_NAME,
    dt_hours=DT,
    epochs=100,
    batch_size=4,
    lr=2e-4,
    d_lr=1e-4,
    d_lr_follow_g=True,
    d_lr_ratio=D_LR_RATIO,
    d_skip_threshold=D_SKIP_THRESHOLD,
    use_cosine_lr=USE_COSINE_LR,
    iD=iD,
    device=DEVICE,
    loss_mode=LOSS_MODE,
    loss_metric=LOSS_METRIC,
    loss_spatial_mode=LOSS_SPATIAL_MODE,
    loss_gaussian_sigma=LOSS_GAUSSIAN_SIGMA,
    protect_existing_run=PROTECT_EXISTING_RUN,
    preview_loss_mask=PREVIEW_LOSS_MASK,
    beta=BETA,
    lambda_perc=LAMBDA_PERC,
    perceptual_backend=PERCEPTUAL_BACKEND,
    lpips_net=LPIPS_NET,
    perceptual_weights=PERCEPTUAL_WEIGHTS,
    perceptual_warmup_epochs=PERCEPTUAL_WARMUP_EPOCHS,
    warmup_epochs=WARMUP_EPOCHS,
    checkpoint_subdir="Models/adv_no",
    isolate_runs=True,
    channel_names=CHANNEL_NAMES,
    viz_every=VIZ_EVERY,
    viz_samples=VIZ_SAMPLES,
    viz_seed=VIZ_SEED,
    viz_residual_metric=VIZ_RESIDUAL_METRIC,
    viz_gaussian_circles=VIZ_GAUSSIAN_CIRCLES,
    viz_gaussian_circle_radii=VIZ_GAUSSIAN_CIRCLE_RADII,
    viz_gaussian_inner_circle_radii=VIZ_GAUSSIAN_INNER_CIRCLE_RADII,
    viz_gaussian_fig_scale=VIZ_GAUSSIAN_FIG_SCALE,
    show_inline_plots=SHOW_INLINE_PLOTS,
)

RUN_DIR.mkdir(parents=True, exist_ok=True)
config_path = RUN_DIR / f"train_config_{RUN_NAME}.json"
config_path.write_text(json.dumps(asdict(train_config), indent=2, default=str))
print(f"Component: {COMPONENT} | iD={iD}")
print(f"Run directory: {RUN_DIR}")
print(f"Wrote config:  {config_path}")
print(f"  data: {DATA_PREFIX} | loss_metric={LOSS_METRIC}")
if LOSS_MODE == "l1":
    print("  loss mode: L1-only (no perceptual / discriminator)")
elif LOSS_MODE == "adv_no":
    perc_name = f"LPIPS ({LPIPS_NET})" if PERCEPTUAL_BACKEND == "lpips" else "VGG-19"
    if PERCEPTUAL_WARMUP_EPOCHS == 0 and WARMUP_EPOCHS == 0:
        print(f"  loss mode: adv-NO — L1 + {perc_name} + adversarial from epoch 1")
    else:
        print(
            f"  loss mode: adv-NO — L1 through ep {PERCEPTUAL_WARMUP_EPOCHS}, "
            f"+{perc_name} from ep {PERCEPTUAL_WARMUP_EPOCHS + 1}, "
            f"+adv from ep {WARMUP_EPOCHS + 1}"
        )
else:
    print(f"  loss mode: {LOSS_MODE!r}")
if LOSS_SPATIAL_MODE == "full":
    spatial_note = "uniform over whole valid field"
elif LOSS_SPATIAL_MODE == "gaussian_center":
    spatial_note = f"gaussian_center (σ={LOSS_GAUSSIAN_SIGMA}) — state⊙G"
else:
    spatial_note = LOSS_SPATIAL_MODE
print(f"  loss spatial: {spatial_note}")
viz_note = "samples_gauss_circles/" if VIZ_GAUSSIAN_CIRCLES else "samples/"
print(f"  viz every {VIZ_EVERY} ep -> viz/{viz_note}")


Component: v | iD=1
Run directory: Models/adv_no/adv_no_v_16_19_dt96_lpips_full_l1_v
Wrote config:  Models/adv_no/adv_no_v_16_19_dt96_lpips_full_l1_v/train_config_adv_no_v_16_19_dt96_lpips_full_l1_v.json
  data: data/sample_16_19_smallfixwin_v_dt96 | loss_metric=components
  loss mode: L1-only (no perceptual / discriminator)
  loss spatial: uniform over whole valid field
  viz every 5 ep -> viz/samples/


In [6]:
from src.train.adv_no_train import _run_has_artifacts
from src.train.fix_window_viz import save_loss_mask_preview

if PROTECT_EXISTING_RUN and _run_has_artifacts(RUN_DIR, RUN_NAME):
    raise FileExistsError(
        f"Run already exists: {RUN_DIR}\n"
        "Change RUN_TAG to a new name before preview / training."
    )
elif _run_has_artifacts(RUN_DIR, RUN_NAME):
    print(f"Note: {RUN_DIR} exists — preview/training will overwrite prior results.")

if PREVIEW_LOSS_MASK:
    preview_path = save_loss_mask_preview(
        DATA_PREFIX,
        RUN_DIR / "viz" / "loss_mask_preview.png",
        sample_index=0,
        loss_spatial_mode=LOSS_SPATIAL_MODE,
        loss_gaussian_sigma=LOSS_GAUSSIAN_SIGMA,
        channel_names=CHANNEL_NAMES,
        show=SHOW_INLINE_PLOTS,
    )
    print(f"Loss mask preview: {preview_path}")
    if LOSS_SPATIAL_MODE == "full":
        print("  full mode: G=1 everywhere — loss treats all valid pixels equally")
    else:
        print(f"  gaussian_center: σ={LOSS_GAUSSIAN_SIGMA}px — center weighted more than edges")
else:
    print("Loss mask preview skipped (PREVIEW_LOSS_MASK=False)")


Loss mask preview: Models/adv_no/adv_no_v_16_19_dt96_lpips_full_l1_v/viz/loss_mask_preview.png
  full mode: G=1 everywhere — loss treats all valid pixels equally


In [7]:
log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)
log_path = log_dir / f"train_{RUN_NAME}.log"

class _Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()

    def flush(self):
        for stream in self.streams:
            stream.flush()


if DETACHED:
    import subprocess
    cmd = [sys.executable, "scripts/train_adv_no_fix_window.py", "--run-name", RUN_NAME]
    with open(log_path, "w") as log_file:
        proc = subprocess.Popen(
            cmd,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            cwd=Path.cwd(),
        )
    print(f"Started background training PID {proc.pid}")
    print(f"Log: {log_path}")
    print(f"Monitor: tail -f {log_path}")
else:
    print(f"Foreground training ({COMPONENT}) — log: {log_path}")
    print("Stop anytime: Kernel → Interrupt (or ■ button)\n")
    _stdout = sys.stdout
    with open(log_path, "w") as log_file:
        sys.stdout = _Tee(_stdout, log_file)
        try:
            run_adv_no_training(train_config)
        except KeyboardInterrupt:
            print("\nNotebook interrupt received — training stopped.")
        finally:
            sys.stdout = _stdout
    print(f"\nLog saved: {log_path}")


Foreground training (v) — log: logs/train_adv_no_v_16_19_dt96_lpips_full_l1_v.log
Stop anytime: Kernel → Interrupt (or ■ button)



Dataset: data/sample_16_19_smallfixwin_v_dt96
Samples: 1461 | crop 128 x 128
Using device: cuda:7
Loss mode: l1
Channels: iD=1 (v)
Reconstruction metric: components (uv L1)
Loss spatial focus: full valid field
lambda_perc: 0.1 | beta: 0.1
Checkpoints: Models/adv_no/adv_no_v_16_19_dt96_lpips_full_l1_v
History:     Models/adv_no/adv_no_v_16_19_dt96_lpips_full_l1_v/history/training_history_adv_no_v_16_19_dt96_lpips_full_l1_v.pt
Viz:         Models/adv_no/adv_no_v_16_19_dt96_lpips_full_l1_v/viz (every 5 epoch(s), fixed val indices [57])
Loss mask preview: Models/adv_no/adv_no_v_16_19_dt96_lpips_full_l1_v/viz/loss_mask_preview.png


/opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/src/train/fix_window_viz.py:266: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc="upper right", fontsize=8)


epoch 1 | val uv L1 0.463048 | (improvement of inf)
Epoch 0001 [L1-only] | Train: 0.492924 (uv L1 0.492924, perc 0.000000, adv 0.000000, D 0.000000) | Val uv L1: 0.463048 (perc 0.000000) | G LR: 2.00e-04
  updated curves -> Models/adv_no/adv_no_v_16_19_dt96_lpips_full_l1_v/viz/curves
epoch 2 | val uv L1 0.451216 | (improvement of 0.011832)
Epoch 0002 [L1-only] | Train: 0.460180 (uv L1 0.460180, perc 0.000000, adv 0.000000, D 0.000000) | Val uv L1: 0.451216 (perc 0.000000) | G LR: 2.00e-04
  updated curves -> Models/adv_no/adv_no_v_16_19_dt96_lpips_full_l1_v/viz/curves
epoch 3 | val uv L1 0.432697 | (improvement of 0.018518)
Epoch 0003 [L1-only] | Train: 0.445878 (uv L1 0.445878, perc 0.000000, adv 0.000000, D 0.000000) | Val uv L1: 0.432697 (perc 0.000000) | G LR: 2.00e-04
  updated curves -> Models/adv_no/adv_no_v_16_19_dt96_lpips_full_l1_v/viz/curves
epoch 4 | val uv L1 0.420425 | (improvement of 0.012272)
Epoch 0004 [L1-only] | Train: 0.433135 (uv L1 0.433135, perc 0.000000, adv 0.0

In [ ]:
import matplotlib.pyplot as plt
import torch

history_path = RUN_DIR / "history" / f"training_history_{RUN_NAME}.pt"
state_path = RUN_DIR / f"train_state_{RUN_NAME}.json"
log_path = Path(f"logs/train_{RUN_NAME}.log")
plot_path = Path("PNG") / f"adv_no_training_loss_curve_{RUN_NAME}.png"
recon_label = f"{COMPONENT} L1"

if not history_path.exists():
    msg = [f"No history yet: {history_path}"]
    if state_path.exists():
        state = json.loads(state_path.read_text())
        msg.append(
            f"Training status: {state.get('status')} | "
            f"last completed epoch: {state.get('last_completed_epoch', 0)}/{state.get('epochs_planned', '?')}"
        )
    if log_path.exists():
        tail = log_path.read_text().splitlines()[-5:]
        msg.append("Recent log lines:")
        msg.extend(f"  {line}" for line in tail)
    msg.append("Re-run this cell after epoch 1 finishes.")
    print("\n".join(msg))
else:
    hist = torch.load(history_path, map_location="cpu", weights_only=False)
    epochs = hist["epoch"]

    if not epochs:
        print("History file exists but is empty. Wait for epoch 1 to finish, then re-run.")
    else:
        fig, ax = plt.subplots(1, 3, figsize=(14, 4))
        ax[0].plot(epochs, hist["train_l1"], label=f"Train {recon_label}")
        ax[0].plot(epochs, hist["val_l1"], label=f"Val {recon_label}")
        ax[0].set_title(f"Reconstruction ({recon_label})")
        ax[0].legend()
        ax[0].grid(True, alpha=0.3)

        use_adv_no = LOSS_MODE == "adv_no"
        perc_warmup = PERCEPTUAL_WARMUP_EPOCHS if use_adv_no else 0
        adv_warmup = WARMUP_EPOCHS if use_adv_no else 0
        perc_title = f"LPIPS ({LPIPS_NET})" if PERCEPTUAL_BACKEND == "lpips" else "VGG-19"

        if use_adv_no and any(hist.get("train_perceptual", [])):
            ax[1].plot(epochs, hist["train_perceptual"], label="Train perc")
            ax[1].plot(epochs, hist["val_perceptual"], label="Val perc")
            if perc_warmup > 0:
                ax[1].axvline(perc_warmup + 0.5, color="#2ca02c", linestyle=":", alpha=0.7)
            ax[1].set_title(f"Perceptual ({perc_title})")
            ax[1].legend()
            ax[1].grid(True, alpha=0.3)
        else:
            ax[1].set_title("Perceptual (adv-NO only)" if not use_adv_no else "Perceptual (after warmup)")
            ax[1].axis("off")

        if use_adv_no and any(hist.get("train_adv", [])):
            ax[2].plot(epochs, hist["train_adv"], label="Train adv")
            ax[2].plot(epochs, hist["train_d"], label="Train D")
            if adv_warmup > perc_warmup:
                ax[2].axvline(adv_warmup + 0.5, color="#888888", linestyle=":", alpha=0.7)
            ax[2].set_title("Adversarial terms")
            ax[2].legend()
            ax[2].grid(True, alpha=0.3)
        else:
            ax[2].set_title("Adversarial (adv-NO only)" if not use_adv_no else "Adversarial (after warmup)")
            ax[2].axis("off")

        plt.tight_layout()
        plot_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Best val {recon_label}: {hist['best_val_loss']:.6f} at epoch {hist['best_epoch']}")
        print(f"Run dir: {RUN_DIR}")
        print(f"Saved plot: {plot_path}")
        curve_png = RUN_DIR / "viz" / "curves" / "loss_overview.png"
        if curve_png.exists():
            print(f"Training curves: {curve_png}")
        if VIZ_GAUSSIAN_CIRCLES:
            circle_png = RUN_DIR / "viz" / "curves" / "circle_l1_overview.png"
            if circle_png.exists():
                print(f"Circle L1 curves: {circle_png}")
